In [ ]:
import json
import re
from datetime import datetime, date
from icalendar import Calendar

def extract_url(text):
    """Extrae el primer enlace http/https que aparezca en la descripción."""
    if not text:
        return None
    # Busca URLs evitando capturar comillas o paréntesis de cierre si es HTML
    match = re.search(r'(https?://[^\s<>"]+)', str(text))
    return match.group(1).rstrip('.,;)') if match else None

def clean_description(text):
    """Limpia etiquetas HTML básicas si el calendario viene de Google Calendar."""
    if not text:
        return ""
    clean = re.sub(r'<[^>]+>', ' ', str(text))
    return ' '.join(clean.split())

def parse_sitges_ics(ics_file_path, output_json_path):
    with open(ics_file_path, 'rb') as f:
        cal = Calendar.from_ical(f.read())

    events_data = []

    for component in cal.walk():
        if component.name == "VEVENT":
            dtstart_prop = component.get('dtstart')
            if not dtstart_prop:
                continue
            
            dtstart = dtstart_prop.dt

            # Filtrar SOLO eventos del mes de Octubre (mes == 10)
            if dtstart.month != 10:
                continue

            sitges_year = dtstart.year
            summary = str(component.get('summary', 'Sin título')).strip()
            location = str(component.get('location', 'Ubicación no especificada')).strip()
            raw_description = str(component.get('description', ''))

            # Extraer link de la descripción o del campo URL nativo del evento
            movie_url = extract_url(raw_description)
            if not movie_url and component.get('url'):
                movie_url = str(component.get('url'))

            # Calcular fecha, hora y duración
            dtend_prop = component.get('dtend')
            dtend = dtend_prop.dt if dtend_prop else None

            if isinstance(dtstart, datetime):
                # Convertir a hora local si tiene zona horaria
                if dtstart.tzinfo is not None:
                    dtstart = dtstart.astimezone()
                date_str = dtstart.strftime('%Y-%m-%d')
                time_str = dtstart.strftime('%H:%M')
                
                duration_min = None
                if isinstance(dtend, datetime):
                    if dtend.tzinfo is not None:
                        dtend = dtend.astimezone()
                    duration_min = int((dtend - dtstart).total_seconds() // 60)
                    end_time_str = dtend.strftime('%H:%M')
                else:
                    end_time_str = None
            elif isinstance(dtstart, date):
                # Evento de día completo
                date_str = dtstart.strftime('%Y-%m-%d')
                time_str = "Todo el día"
                end_time_str = None
                duration_min = None

            events_data.append({
                "sitges_year": sitges_year,
                "movie_title": summary,
                "date": date_str,
                "start_time": time_str,
                "end_time": end_time_str,
                "duration_min": duration_min,
                "location": location if location else "Sala por confirmar",
                "movie_url": movie_url,
                "description": clean_description(raw_description)
            })

    # Ordenar cronológicamente por fecha y hora
    events_data.sort(key=lambda x: (x['date'], x['start_time']))

    with open(output_json_path, 'w', encoding='utf-8') as f:
        json.dump(events_data, f, ensure_ascii=False, indent=4)

    print(f"¡Hecho! Se han exportado {len(events_data)} sesiones de octubre a '{output_json_path}'.")